# Characteristic-Managed Momentum (CMM) — Interactive Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DorLev165/cmm-momentum-replication/blob/main/notebooks/demo.ipynb)

This notebook demonstrates the **Characteristic-Managed Momentum (CMM)** model — a PyTorch neural network that learns flexible softmax weights over past daily returns to predict next-month cross-sectional stock returns.

**Architecture:**
- 3-layer FFN (32 → 16 → 8) with softmax output
- Ensemble of 5 random seeds (averaged predictions)
- Trained to predict next-month returns cross-sectionally

**Pipeline:**
1. Fetch S&P 500 price data via `yfinance`
2. Build characteristics + daily return features per stock-month
3. Run an **expanding-window** training loop (3 windows shown here)
4. Sort into HML (High-Minus-Low) decile portfolios
5. Plot cumulative returns and summary statistics

> **Note:** This demo uses `yfinance` (survivorship-biased, S&P 500 only) and 3 expanding windows with reduced epochs for speed. The full replication uses WRDS/JKP data and ~12+ windows.

## 1. Install Dependencies

In [ ]:
# Install all required packages
!pip install -q torch numpy pandas scikit-learn yfinance matplotlib lxml

## 2. Clone the Repository

In [ ]:
import os

REPO_URL = "https://github.com/DorLev165/cmm-momentum-replication"
REPO_DIR = "cmm-momentum-replication"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
    print(f"Cloned into {REPO_DIR}/")
else:
    print(f"Repository already present at {REPO_DIR}/")

# Add the project root to the Python path so `import cmm` works
import sys
sys.path.insert(0, os.path.abspath(REPO_DIR))
print("sys.path updated — cmm package is importable")

## 3. Imports and Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# CMM package
from cmm import prepare_cmm_data
from cmm.fetch_data import fetch_sp500_cmm_data
from cmm.training import run_expanding_window
from cmm.portfolio import hml_summary, vol_managed_returns

print("All imports successful.")
print(f"NumPy  {np.__version__}")
print(f"Pandas {pd.__version__}")

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}  |  device: {device}")

## 4. Fetch S&P 500 Data

This cell downloads daily price history for S&P 500 constituents via `yfinance` and builds the feature arrays used by the model:
- **characteristics** — 11 price-based signals (momentum, volatility, skewness, etc.)
- **daily_returns** — 231 daily log-returns per stock-month (`t-252` to `t-22`, skipping the reversal lag)
- **next_month_returns** — the label each observation is trained against

In [ ]:
# Demo window: 2010-2022. Keeping it to ~12 years balances download time
# vs. having enough data for 3 meaningful expanding windows.
START_DATE = "2010-01-01"
END_DATE   = "2022-12-31"

# Cap at 150 tickers to keep the Colab download fast (~60-90 seconds).
# Remove or increase MAX_TICKERS to use more stocks (slower).
MAX_TICKERS = 150

print(f"Fetching S&P 500 data: {START_DATE} to {END_DATE}  (max {MAX_TICKERS} tickers)")
print("This takes about 1-2 minutes on a fresh Colab instance ...")

(
    characteristics,
    daily_returns,
    next_month_returns,
    dates,
    timestamps,
    tickers_arr,
    market_cap,
    is_nyse,
    regime_signal,
    industries,
) = fetch_sp500_cmm_data(
    start_date=START_DATE,
    end_date=END_DATE,
    n_char=11,
    max_tickers=MAX_TICKERS,
)

ts = pd.to_datetime(timestamps)
print(f"\nData summary:")
print(f"  Stock-month observations : {len(timestamps):,}")
print(f"  Date range               : {ts.min().date()} to {ts.max().date()}")
print(f"  Characteristics shape    : {characteristics.shape}")
print(f"  Daily returns shape      : {daily_returns.shape}  (231 lags per obs)")
print(f"  Next-month returns shape : {next_month_returns.shape}")

## 5. Run Expanding-Window CMM Training

The expanding-window loop:
1. **Train** on all data up to `train_end_year`
2. **Predict** on the following 2-year test period
3. **Expand** the training window by 2 years and repeat

For this demo we run **3 windows** (`n_windows_limit=3`) with reduced epochs so each window finishes in roughly 1-3 minutes on Colab CPU (or faster on GPU).

| Setting | Demo value | Full replication |
|---|---|---|
| Epochs | 30 | 100 |
| Ensemble seeds | 2 | 5 |
| Windows | 3 | ~6 (for 2010-2022) |
| Batch size | 4096 | 4096 |

In [ ]:
# --- Demo hyper-parameters (edit these to explore) ---
DEMO_EPOCHS     = 30    # reduce to 10 for a very fast smoke-test; use 100 for paper fidelity
DEMO_ENSEMBLES  = 2     # 2 seeds keeps runtime sane; paper uses 5
N_WINDOWS_LIMIT = 3     # number of expanding windows to run; None = all

# Derive training start/end from the data we actually have
min_year = int(pd.to_datetime(timestamps).year.min())
max_year = int(pd.to_datetime(timestamps).year.max())

# Use a 4-year initial training window so the first window has enough data
train_start   = min_year
initial_train_end = min_year + 3   # e.g. 2010-2013 -> test 2014-2015

print("CMM expanding-window demo")
print(f"  Data range        : {min_year}-{max_year}")
print(f"  Initial train     : {train_start}-{initial_train_end}")
print(f"  Windows to run    : {N_WINDOWS_LIMIT}")
print(f"  Epochs per window : {DEMO_EPOCHS}")
print(f"  Ensemble seeds    : {DEMO_ENSEMBLES}")
print()

results = run_expanding_window(
    characteristics,
    daily_returns,
    next_month_returns,
    dates,
    timestamps,
    tickers_arr,
    market_cap,
    is_nyse=is_nyse,
    regime_signal=regime_signal,
    industries=industries,
    # Model architecture (paper-faithful NN3)
    n_char=11,
    n_ret=231,
    hidden_sizes=(32, 16, 8),
    dropout=0.0,
    layer_norm=False,
    output_init_scale=1.5,
    # Optimisation
    epochs=DEMO_EPOCHS,
    batch_size=4096,
    learning_rate=1e-3,
    weight_decay=0.0,
    grad_clip_norm=1.0,
    lr_schedule="cosine",
    warmup_epochs=3,
    # Early stopping
    early_stopping_patience=10,
    min_epochs=10,
    # Loss
    loss_fn="mse",
    # Ensembling
    n_ensembles=DEMO_ENSEMBLES,
    # Expanding-window settings
    train_start_year=train_start,
    initial_train_end_year=initial_train_end,
    val_pct=0.3,
    # Keep regime ensemble off for yfinance mode (regime signal is all-zeros)
    use_regime_ensemble=False,
    use_mv_portfolio=False,
    scale_features=True,
    # Don't write plots to disk in Colab (we'll render them inline)
    weights_plot_dir=None,
    partial_save_dir=None,
)

# Optionally limit to N_WINDOWS_LIMIT
if N_WINDOWS_LIMIT and len(results) > N_WINDOWS_LIMIT:
    results = results[:N_WINDOWS_LIMIT]
    print(f"  (truncated to first {N_WINDOWS_LIMIT} windows for demo)")

print(f"\nCompleted {len(results)} expanding window(s).")

## 6. Per-Window Results

In [ ]:
if not results:
    print("No results — check that the data range covers at least 5 years.")
else:
    print("-" * 75)
    print(f"{'Window':<28}  {'VW Sharpe':>10}  {'AnnRet%':>8}  {'MDD%':>7}  {'IC':>8}")
    print("-" * 75)
    for r in results:
        stats = r.get("hml_stats", {})
        sh  = stats.get("sharpe", float("nan"))
        ar  = stats.get("ann_ret", float("nan")) * 100
        mdd = stats.get("mdd", float("nan")) * 100
        ic  = r.get("ic", float("nan"))
        label = f"{r['train_start']}-{r['train_end']} / {r['test_start']}-{r['test_end']}"
        print(f"{label:<28}  {sh:>+10.2f}  {ar:>+8.1f}  {mdd:>+7.1f}  {ic:>+8.4f}")
    print("-" * 75)

## 7. Pooled HML Statistics

In [ ]:
# Concatenate all test-period HML return series
hml_frames = [
    r["hml_df"][["date", "HML_ret"]].set_index("date")
    for r in results
    if "hml_df" in r and len(r["hml_df"]) > 0
]

if not hml_frames:
    print("No HML return data available.")
else:
    all_hml = pd.concat(hml_frames).sort_index()
    all_hml = all_hml[~all_hml.index.duplicated(keep="first")]

    pooled = hml_summary(all_hml["HML_ret"])

    print("Pooled HML (value-weighted, all test windows):")
    print(f"  Sharpe ratio     : {pooled.get('sharpe', float('nan')):+.3f}")
    print(f"  Annualised return: {pooled.get('ann_ret', float('nan'))*100:+.2f}%")
    print(f"  Annualised vol   : {pooled.get('vol', float('nan'))*100:.2f}%")
    print(f"  Max drawdown     : {pooled.get('mdd', float('nan'))*100:+.2f}%")
    print(f"  Months           : {pooled.get('n_months', 0)}")
    mean_ic = np.mean([r.get("ic", 0) for r in results])
    std_ic  = np.std( [r.get("ic", 0) for r in results])
    print(f"  Mean IC          : {mean_ic:+.4f}  (std {std_ic:.4f})")

## 8. Plots

### 8a. Cumulative Return and Rolling Variance

In [ ]:
if "all_hml" not in dir() or all_hml is None or len(all_hml) < 2:
    print("Not enough HML data to plot.")
else:
    cumret  = (1 + all_hml["HML_ret"]).cumprod()
    roll_var = all_hml["HML_ret"].rolling(12, min_periods=1).var() * 12  # annualised

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    fig.suptitle("CMM HML Portfolio — Demo Run", fontsize=13, fontweight="bold")

    ax1.plot(cumret.index, cumret.values, color="steelblue", linewidth=1.8, label="HML cumret")
    ax1.axhline(1.0, color="gray", linestyle="--", alpha=0.6)
    ax1.set_ylabel("Growth of $1")
    ax1.set_title("Cumulative return — HML (long D10, short D1)")
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)

    ax2.fill_between(roll_var.index, roll_var.values, alpha=0.4, color="coral")
    ax2.plot(roll_var.index, roll_var.values, color="coral", linewidth=1.2)
    ax2.set_ylabel("Variance (ann.)")
    ax2.set_xlabel("Year")
    ax2.set_title("Rolling 12-month variance (annualised)")
    ax2.grid(True, alpha=0.3)

    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    plt.tight_layout()
    plt.show()

### 8b. Decile Return Bar Chart

In [ ]:
# Aggregate average monthly returns by decile across all test windows
decile_frames = []
for r in results:
    hdf = r.get("hml_df")
    if hdf is None or hdf.empty:
        continue
    # Each hml_df has columns: date, D1_ret, D2_ret, ..., D10_ret, HML_ret
    dcols = [c for c in hdf.columns if c.startswith("D") and c.endswith("_ret")]
    if dcols:
        decile_frames.append(hdf[dcols])

if decile_frames:
    all_deciles = pd.concat(decile_frames, ignore_index=True)
    mean_by_decile = all_deciles.mean() * 100  # monthly % return

    # Sort by decile number
    order = [f"D{i}_ret" for i in range(1, 11) if f"D{i}_ret" in mean_by_decile.index]
    mean_by_decile = mean_by_decile.reindex(order)
    labels = [str(i) for i in range(1, len(order) + 1)]

    fig, ax = plt.subplots(figsize=(9, 4))
    colors = ["tomato" if v < 0 else "steelblue" for v in mean_by_decile.values]
    ax.bar(labels, mean_by_decile.values, color=colors, edgecolor="white", linewidth=0.5)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Decile (1 = lowest CMM signal, 10 = highest)")
    ax.set_ylabel("Avg monthly return (%)")
    ax.set_title("Average monthly return by CMM decile (value-weighted, pooled)")
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Decile return columns not found in hml_df — skipping bar chart.")
    print("Available columns:", results[0]["hml_df"].columns.tolist() if results else "(no results)")

### 8c. Volatility-Managed Overlay

Scales monthly HML exposure inversely to trailing 6-month variance, targeting 12% annualised vol (Daniel-Moskowitz 2016 / Barroso-Santa-Clara 2015). This is the lightest-weight way to improve crash resistance without modifying the cross-sectional model.

In [ ]:
if "all_hml" not in dir() or all_hml is None or len(all_hml) < 8:
    print("Not enough data for vol-managed overlay (need at least 8 months).")
else:
    scaled, leverage = vol_managed_returns(
        all_hml["HML_ret"],
        target_ann_vol=0.12,
        lookback_months=6,
        max_leverage=3.0,
    )
    scaled_summary = hml_summary(scaled)

    print("Vol-managed HML (target 12% ann vol, trailing 6mo, max 3x leverage):")
    print(f"  Sharpe           : {scaled_summary.get('sharpe', float('nan')):+.3f}")
    print(f"  Annualised return: {scaled_summary.get('ann_ret', float('nan'))*100:+.2f}%")
    print(f"  Annualised vol   : {scaled_summary.get('vol', float('nan'))*100:.2f}%")
    print(f"  Max drawdown     : {scaled_summary.get('mdd', float('nan'))*100:+.2f}%")
    print(f"  Avg leverage     : {leverage.mean():.2f}x")

    # Compare raw vs vol-managed cumulative return
    cumret_raw    = (1 + all_hml["HML_ret"]).cumprod()
    cumret_scaled = (1 + scaled).cumprod()

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(cumret_raw.index,    cumret_raw.values,    color="steelblue", linewidth=1.5, label="Raw HML (VW)")
    ax.plot(cumret_scaled.index, cumret_scaled.values, color="darkorange", linewidth=1.5, linestyle="--", label="Vol-managed HML")
    ax.axhline(1.0, color="gray", linestyle=":", alpha=0.6)
    ax.set_ylabel("Growth of $1")
    ax.set_xlabel("Year")
    ax.set_title("Raw HML vs Vol-managed HML — cumulative return")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    plt.tight_layout()
    plt.show()

## 9. Next Steps

| Task | How |
|---|---|
| Run more windows | Remove `N_WINDOWS_LIMIT` restriction |
| Paper-faithful training | Set `DEMO_EPOCHS=100`, `DEMO_ENSEMBLES=5` |
| Full universe (all tickers) | Set `MAX_TICKERS=None` or remove the parameter |
| Full WRDS/JKP replication | Set `CMM_DATA_SOURCE=jkp` and run `main.py` locally with WRDS credentials |
| Inspect learned weights | Access `results[i]['weights_plot_path']` (enable `weights_plot_dir`) |

**Repository:** [github.com/DorLev165/cmm-momentum-replication](https://github.com/DorLev165/cmm-momentum-replication)